In [35]:
import pandas as pd
from sqlalchemy import text, create_engine
from config import DB_URL

In [36]:
create_apartment_sales_table_sql = '''
CREATE TABLE IF NOT EXISTS seoul_apartment_sales (
    month INT,
    district VARCHAR(50),
    units_sold INT,
    PRIMARY KEY (month, district)
)
'''

insert_apartment_sales_sql = '''
INSERT INTO seoul_apartment_sales (
    month,
    district,
    units_sold
) VALUES (
    :month,
    :district,
    :units_sold
)
ON CONFLICT (month, district) DO NOTHING
'''

In [38]:
def get_apartment_sales_df():
    df = pd.read_csv('/workspaces/korea-real-estate-population-movement/data/seoul_apartments_sold.csv')
    df = df[2:].reset_index(drop=True).drop(columns=[
        "자치구별(1)"
        ]).rename(columns={
        "자치구별(2)":"district"
        })

    df = df.melt(
        id_vars=["district"],
        var_name="month",
        value_name="units_sold")

    df["month"] = pd.to_datetime(
        df["month"].str.strip(),
        format="%Y. %m"
    ).dt.strftime("%Y%m").astype(int)

    return df.to_dict(orient="records")


def main():
    engine = create_engine(DB_URL)
    records = get_apartment_sales_df()

    with engine.begin() as conn:
        conn.execute(text(create_apartment_sales_table_sql))
        conn.execute(text(insert_apartment_sales_sql),records)

        
main()